[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/svngoku/cybersup-deep-learning-intro/blob/main/notebooks/formateur/01_mlp_backprop_corrige.ipynb)


# TP 01 — MLP et rétropropagation en NumPy
**Durée : 150 minutes · Master 2 IA · CPU**

Objectif : relier calcul direct, dérivation, contrôle numérique et généralisation.
La base est exécutable ; les exercices demandent d'expliquer, de modifier et de comparer.
Ne regardez les scores test qu'après avoir fixé votre configuration.

Découpage : lecture 20 min, gradients 40 min, contrôle 25 min, expériences 40 min, restitution 25 min.

**VERSION FORMATEUR — réponses et points de contrôle en fin de notebook.**

## Préparation de l'environnement

Exécuter cette cellule en premier dans un runtime Python 3 sur **CPU**. Les bibliothèques déjà installées sont conservées ; seules celles qui manquent sont ajoutées. Aucun clonage, fichier voisin ou montage Drive n'est nécessaire. Enregistrer une copie du notebook avant de commencer. Si le lien vers le dépôt privé est inaccessible, importer le fichier fourni dans le pack étudiant.


In [ ]:
# Cellule autonome : les bibliothèques déjà présentes dans Colab sont conservées.
import importlib.util
import json
import os
import platform
import subprocess
import sys

DEPENDENCIES = {
    "numpy": "numpy==2.2.6",
    "sklearn": "scikit-learn==1.7.2",
    "torch": "torch==2.8.0",
    "matplotlib": "matplotlib==3.10.6",
}
missing = [pin for module, pin in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des bibliothèques absentes :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import matplotlib
import numpy as np
import sklearn
import torch

ENVIRONMENT = {
    "runtime": "google-colab" if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ else "python-local-ou-ci",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
    "matplotlib": matplotlib.__version__,
    "device": "cpu",
    "cuda_available": torch.cuda.is_available(),
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))
print("Les cinq TP utilisent le CPU, même si un GPU est disponible.")


In [ ]:
import os, copy, math, random
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device('cpu')
print('Référence : CPU | PyTorch', torch.__version__, '| graine', SEED)

In [ ]:
X,y=make_moons(n_samples=900,noise=0.18,random_state=SEED)
Xt,Xtest,yt,ytest=train_test_split(X,y,test_size=0.15,stratify=y,random_state=SEED)
Xtr,Xval,ytr,yval=train_test_split(Xt,yt,test_size=0.15/0.85,stratify=yt,random_state=SEED)
mu=Xtr.mean(0); std=Xtr.std(0)
Xtr=(Xtr-mu)/std; Xval=(Xval-mu)/std; Xtest=(Xtest-mu)/std
rng=np.random.default_rng(SEED)
params={'W1':rng.normal(size=(2,16))*np.sqrt(2/2),'b1':np.zeros(16),
        'W2':rng.normal(size=(16,2))*np.sqrt(2/16),'b2':np.zeros(2)}
# Ici les poids NumPy sont stockés entrée × sortie, contrairement à nn.Linear.
assert sum(v.size for v in params.values())==82
print('Train/val/test :',len(ytr),len(yval),len(ytest))

## 1. Annoter chaque gradient
Les exemples sont des lignes. Le gradient initial contient déjà la division par le batch.
Expliquez chaque produit matriciel et vérifiez sa forme avant d'exécuter la cellule.

In [ ]:
def forward(X,p):
    z1=X@p['W1']+p['b1']; h=np.maximum(z1,0)
    logits=h@p['W2']+p['b2']
    shifted=logits-logits.max(axis=1,keepdims=True)
    logp=shifted-np.log(np.exp(shifted).sum(axis=1,keepdims=True))
    return logp,(X,z1,h)

def loss_grad(X,y,p):
    logp,(X,z1,h)=forward(X,p); probs=np.exp(logp)
    loss=-logp[np.arange(len(y)),y].mean()
    delta=probs.copy(); delta[np.arange(len(y)),y]-=1; delta/=len(y)
    hidden=(delta@p['W2'].T)*(z1>0)
    grads={'W2':h.T@delta,'b2':delta.sum(0),'W1':X.T@hidden,'b1':hidden.sum(0)}
    return loss,grads

def score(X,y,p):
    lp,_=forward(X,p)
    return -lp[np.arange(len(y)),y].mean(),(lp.argmax(1)==y).mean()

smallX=Xtr[:8]; smally=ytr[:8]
loss,grads=loss_grad(smallX,smally,params)
eps=1e-5; errors={}
for name,value in params.items():
    numeric=np.zeros_like(value)
    for idx in np.ndindex(value.shape):
        saved=value[idx]
        value[idx]=saved+eps; plus=score(smallX,smally,params)[0]
        value[idx]=saved-eps; minus=score(smallX,smally,params)[0]
        value[idx]=saved; numeric[idx]=(plus-minus)/(2*eps)
    errors[name]=float(np.linalg.norm(numeric-grads[name])/(np.linalg.norm(numeric)+np.linalg.norm(grads[name])+1e-12))
print('Erreurs relatives :',errors)
assert max(errors.values())<1e-5

In [ ]:
initial=copy.deepcopy(params); best=copy.deepcopy(params); best_val=float('inf'); history=[]
LR=0.15; EPOCHS=250
for epoch in range(EPOCHS):
    loss,g=loss_grad(Xtr,ytr,params)
    for name in params: params[name]-=LR*g[name]
    tr=score(Xtr,ytr,params); va=score(Xval,yval,params); history.append((*tr,*va))
    if va[0]<best_val: best_val=va[0]; best=copy.deepcopy(params)
history=np.array(history); params=best
plt.plot(history[:,0],label='train');plt.plot(history[:,2],label='validation')
plt.xlabel('Époque');plt.ylabel('Entropie croisée');plt.legend();plt.show()
print('Validation du checkpoint choisi :',score(Xval,yval,params))

## 2. Expériences à réaliser — 40 min
1. Repartir de `initial` et comparer les taux 0,01 ; 0,15 ; 1,0 à budget identique.
2. Remplacer ReLU par tanh **et adapter sa dérivée** ; refaire le contrôle numérique.
3. Comparer 4, 16 et 64 unités, sans utiliser le test pour choisir.
4. Distinguer dans le rapport les paramètres stockés entrée×sortie et ceux de nn.Linear.

Écrire ici : hypothèse, configuration, résultat validation, explication et limite.

In [ ]:
# Bilan terminal : aucune modification de modèle après lecture de ce score.
test_loss,test_acc=score(Xtest,ytest,params)
RESULT={'tp':1,'gradient_max_relative_error':max(errors.values()),'test_loss':test_loss,'test_accuracy':test_acc}
print(RESULT)

## Corrigé — points de contrôle
- W1 a la forme 2×16 et W2 la forme 16×2 dans ce notebook. Les gradients ont la même forme.
- Le facteur 1/B figure dans delta ; le remettre dans h.T@delta serait une erreur.
- Pour tanh, remplacer h par tanh(z1) et la dérivée par 1-h² ; conserver tout le reste.
- LR=1 n'est pas garanti de diverger sur cette petite tâche ; décrire le résultat observé, pas une attente rigide.
- La comparaison de largeur doit repartir d'une initialisation cohérente et annoncer les paramètres (5m+2).
- La normalisation est apprise uniquement sur Xtr. Un score test n'est pas une preuve externe.

Correction du calcul scalaire : a=0,7310586, dL/dw≈−0,105754 ; pour η=0,1, w≈0,510575 et b≈0,005288.

## Exporter les résultats

Exécuter la cellule suivante après le bilan terminal pour enregistrer métriques, graine et versions dans un fichier JSON. Dans Colab, activer `DOWNLOAD_RESULTS` pour le télécharger. Sauvegarder également une copie du notebook avec les sorties et les réponses aux investigations. Les fichiers du runtime sont temporaires.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

DOWNLOAD_RESULTS = False  # @param {type:"boolean"}

def json_scalar(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Valeur non sérialisable : {type(value).__name__}")

result_path = Path("resultats") / f"tp_{RESULT['tp']:02d}.json"
result_path.parent.mkdir(exist_ok=True)
experiment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "environment": ENVIRONMENT,
    "results": RESULT,
}
result_path.write_text(
    json.dumps(experiment, indent=2, ensure_ascii=False, default=json_scalar) + "\n",
    encoding="utf-8",
)
print("Résultats enregistrés :", result_path)

if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print("Hors Colab, récupérer le fichier dans le dossier resultats/.")
    else:
        files.download(str(result_path))
